# 🛍️ Customer Segmentation — Part 4: Insights & Marketing Strategy

**Input:** `data/rfm_clustered.parquet` (from notebook 03)  
**Output:** `outputs/segments_final.csv` — final segment table with marketing recommendations

---

## What we'll cover
1. Load clustered data
2. Name and profile each cluster
3. Build segment personas
4. Revenue opportunity analysis
5. Marketing recommendations per segment
6. Executive summary dashboard
7. Export final deliverable


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.titleweight': 'bold',
})

CLUSTER_COLORS = ['#1D9E75', '#378ADD', '#D85A30', '#7F77DD', '#BA7517']

print('Setup complete ✓')

## 1. Load Clustered Data

In [ ]:
rfm = pd.read_parquet('../data/rfm_clustered.parquet')

N_CLUSTERS = rfm['Cluster'].nunique()
print(f'Customers  : {len(rfm):,}')
print(f'Clusters   : {N_CLUSTERS}')
print(f'Columns    : {rfm.columns.tolist()}')
rfm.head()

## 2. Profile Each Cluster

We analyze the clusters by their average RFM values to understand what each group looks like.

In [ ]:
total_revenue = rfm['Monetary'].sum()

profile = (
    rfm.groupby('Cluster')
    .agg(
        Customers     = ('Customer ID', 'count'),
        Avg_Recency   = ('Recency',    'mean'),
        Median_Recency= ('Recency',    'median'),
        Avg_Frequency = ('Frequency',  'mean'),
        Avg_Monetary  = ('Monetary',   'mean'),
        Total_Revenue = ('Monetary',   'sum'),
        Avg_R_Score   = ('R_Score',    'mean'),
        Avg_F_Score   = ('F_Score',    'mean'),
        Avg_M_Score   = ('M_Score',    'mean'),
        Avg_RFM_Score = ('RFM_Score',  'mean'),
    )
    .round(1)
    .sort_values('Avg_RFM_Score', ascending=False)
)

profile['Revenue_Share_%']   = (profile['Total_Revenue'] / total_revenue * 100).round(1)
profile['Customer_Share_%']  = (profile['Customers'] / len(rfm) * 100).round(1)

print(profile[['Customers','Customer_Share_%','Avg_Recency','Avg_Frequency',
               'Avg_Monetary','Total_Revenue','Revenue_Share_%']].to_string())

## 3. Name the Segments

Based on the RFM profiles above, we assign meaningful business names to each cluster.

> **How to read the profiles:**
> - Low Recency + High Frequency + High Monetary = **Champions**
> - High Recency (bought long ago) + Low Frequency = **Lost / At Risk**
> - Low Recency + Low Frequency = **New Customers**

Adjust the mapping below based on your actual cluster profiles.

In [ ]:
# ── Assign names based on profile analysis ────────────────────────────────────
# Review the profile table above and map cluster IDs to names.
# Example mapping (adjust cluster IDs to match YOUR output):

# Sort clusters by avg RFM score descending to assign names in order
ranked_clusters = profile.index.tolist()  # sorted by Avg_RFM_Score desc

# Default name map based on ranking (best → worst)
default_names = [
    'Champions',
    'Loyal Customers',
    'At Risk',
    'Promising',
    'Lost',
]

segment_name_map = {
    cluster_id: default_names[i]
    for i, cluster_id in enumerate(ranked_clusters[:len(default_names)])
}

rfm['Segment'] = rfm['Cluster'].map(segment_name_map)

print('Segment assignments:')
for k, v in segment_name_map.items():
    n = (rfm['Cluster'] == k).sum()
    print(f'  Cluster {k} → {v} ({n:,} customers)')

## 4. Segment Personas

Each segment gets a **persona** — a business description that non-technical stakeholders can understand.

In [ ]:
PERSONAS = {
    'Champions': {
        'emoji':       '🏆',
        'description': 'Your best customers. Bought recently, buy often, and spend the most.',
        'behavior':    'Brand advocates. Will respond well to loyalty rewards and early access.',
        'risk':        'Low — but can churn if ignored.',
        'color':       '#1D9E75',
    },
    'Loyal Customers': {
        'emoji':       '💎',
        'description': 'Buy regularly and spend well. Not quite Champions yet.',
        'behavior':    'Consistent buyers who respond well to upsell and cross-sell.',
        'risk':        'Low — high lifetime value if nurtured.',
        'color':       '#378ADD',
    },
    'At Risk': {
        'emoji':       '⚠️',
        'description': 'Used to buy often and spend well — but haven\'t purchased recently.',
        'behavior':    'Previously valuable customers drifting away.',
        'risk':        'HIGH — immediate win-back needed.',
        'color':       '#D85A30',
    },
    'Promising': {
        'emoji':       '🌱',
        'description': 'Recent buyers with low frequency — early in their journey.',
        'behavior':    'Have shown interest. Need nurturing to become loyal.',
        'risk':        'Medium — could go either way.',
        'color':       '#7F77DD',
    },
    'Lost': {
        'emoji':       '💤',
        'description': 'Haven\'t bought in a long time and had low engagement.',
        'behavior':    'Largely inactive. Low ROI to target individually.',
        'risk':        'Very high — most won\'t return.',
        'color':       '#888780',
    },
}

print('Personas defined for:', list(PERSONAS.keys()))

In [ ]:
# Print full persona cards
seg_summary = (
    rfm.groupby('Segment')
    .agg(
        Customers    = ('Customer ID', 'count'),
        Avg_Recency  = ('Recency',    'mean'),
        Avg_Frequency= ('Frequency',  'mean'),
        Avg_Monetary = ('Monetary',   'mean'),
        Total_Revenue= ('Monetary',   'sum'),
    )
    .round(1)
)
seg_summary['Revenue_Share_%'] = (
    seg_summary['Total_Revenue'] / total_revenue * 100
).round(1)

print('=' * 65)
for seg, persona in PERSONAS.items():
    if seg not in seg_summary.index:
        continue
    row = seg_summary.loc[seg]
    print(f"\n{persona['emoji']}  {seg}")
    print(f"   Customers     : {int(row['Customers']):,} ({row['Customers']/len(rfm)*100:.1f}% of base)")
    print(f"   Revenue share : {row['Revenue_Share_%']}% (£{row['Total_Revenue']:,.0f})")
    print(f"   Avg recency   : {row['Avg_Recency']:.0f} days")
    print(f"   Avg orders    : {row['Avg_Frequency']:.1f}")
    print(f"   Avg spend     : £{row['Avg_Monetary']:,.0f}")
    print(f"   Description   : {persona['description']}")
    print(f"   Risk level    : {persona['risk']}")
print('=' * 65)

## 5. Revenue Opportunity Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Customer Segment Overview', fontsize=15, fontweight='bold')

seg_order  = [s for s in PERSONAS.keys() if s in seg_summary.index]
seg_colors = [PERSONAS[s]['color'] for s in seg_order]
seg_data   = seg_summary.loc[seg_order]

# Plot 1: Customer count
ax = axes[0]
bars = ax.barh(seg_order[::-1], seg_data.loc[seg_order[::-1], 'Customers'],
               color=seg_colors[::-1], edgecolor='white')
ax.set_title('Customers per Segment')
ax.set_xlabel('Number of Customers')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 5, bar.get_y() + bar.get_height()/2,
            f'{int(w):,}', va='center', fontsize=9)

# Plot 2: Revenue share (donut)
ax = axes[1]
wedges, _, autotexts = ax.pie(
    seg_data['Total_Revenue'],
    labels=None,
    colors=seg_colors,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.78,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2, 'width': 0.6}
)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title('Revenue Share')
legend_patches = [mpatches.Patch(color=PERSONAS[s]['color'], label=s)
                  for s in seg_order]
ax.legend(handles=legend_patches, loc='lower center',
          bbox_to_anchor=(0.5, -0.18), fontsize=8, ncol=2)

# Plot 3: Avg monetary per segment
ax = axes[2]
bars = ax.barh(seg_order[::-1], seg_data.loc[seg_order[::-1], 'Avg_Monetary'],
               color=seg_colors[::-1], edgecolor='white')
ax.set_title('Avg Customer Value (£)')
ax.set_xlabel('Average Monetary (GBP)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'£{x:,.0f}'))
for bar in bars:
    w = bar.get_width()
    ax.text(w + 5, bar.get_y() + bar.get_height()/2,
            f'£{int(w):,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../outputs/figures/segment_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Bubble chart: Size vs Value vs Recency ────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))

for seg in seg_order:
    if seg not in seg_summary.index:
        continue
    row   = seg_summary.loc[seg]
    color = PERSONAS[seg]['color']
    size  = row['Customers'] * 3

    ax.scatter(
        row['Avg_Recency'],
        row['Avg_Monetary'],
        s=size, color=color, alpha=0.75, edgecolors='white', linewidth=1.5
    )
    ax.annotate(
        f"{seg}\n({int(row['Customers']):,})",
        xy=(row['Avg_Recency'], row['Avg_Monetary']),
        xytext=(10, 10), textcoords='offset points',
        fontsize=10, fontweight='bold', color=color
    )

ax.set_xlabel('Avg Recency (days since last purchase) — Lower is better →', fontsize=11)
ax.set_ylabel('Avg Monetary Value (£) — Higher is better →', fontsize=11)
ax.set_title('Segment Map: Recency vs Value\n(bubble size = number of customers)',
             fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.invert_xaxis()  # recent = right side

plt.tight_layout()
plt.savefig('../outputs/figures/segment_bubble_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Marketing Recommendations

Each segment gets a tailored marketing strategy.

In [ ]:
STRATEGY = {
    'Champions': {
        'goal':       'Retain & grow',
        'channels':   ['Email VIP program', 'Loyalty rewards', 'Referral program'],
        'message':    'Thank you for being a top customer — here\'s early access / exclusive offer',
        'frequency':  'Monthly exclusive content + quarterly rewards',
        'budget':     'High — protect this revenue at all cost',
        'kpi':        'Churn rate, repeat purchase rate, NPS',
    },
    'Loyal Customers': {
        'goal':       'Upsell & deepen loyalty',
        'channels':   ['Email personalization', 'Push notifications', 'Cross-sell campaigns'],
        'message':    'Products recommended based on your history + loyalty points update',
        'frequency':  'Bi-weekly personalized recommendations',
        'budget':     'Medium-high — good ROI for upsell',
        'kpi':        'AOV growth, product category expansion, retention rate',
    },
    'At Risk': {
        'goal':       'Win back before they\'re gone',
        'channels':   ['Email win-back sequence', 'Retargeting ads', 'SMS if opted-in'],
        'message':    'We miss you — here\'s 15% off your next order',
        'frequency':  '3-email sequence over 2 weeks, then suppress',
        'budget':     'Medium — time-sensitive, high value if recovered',
        'kpi':        'Win-back rate, revenue recovered, unsubscribe rate',
    },
    'Promising': {
        'goal':       'Convert to loyal buyers',
        'channels':   ['Welcome email series', 'Educational content', 'First repeat incentive'],
        'message':    'Here\'s what other customers love + 10% off your second order',
        'frequency':  'Weekly for first month, then bi-weekly',
        'budget':     'Medium — invest now to build loyalty',
        'kpi':        'Second purchase rate, 90-day retention',
    },
    'Lost': {
        'goal':       'Reactivate cheaply or sunset',
        'channels':   ['One-off reactivation email', 'Paid retargeting (low bid)'],
        'message':    'It\'s been a while — here\'s what\'s new + a comeback offer',
        'frequency':  'Single reactivation email. If no response, suppress.',
        'budget':     'Low — minimal spend, focus on other segments',
        'kpi':        'Reactivation rate, cost per reactivation',
    },
}

print('='*65)
print('MARKETING STRATEGY BY SEGMENT')
print('='*65)

for seg, strat in STRATEGY.items():
    if seg not in seg_summary.index:
        continue
    print(f"\n{PERSONAS[seg]['emoji']} {seg}")
    print(f"   Goal      : {strat['goal']}")
    print(f"   Channels  : {', '.join(strat['channels'])}")
    print(f"   Message   : {strat['message']}")
    print(f"   Frequency : {strat['frequency']}")
    print(f"   Budget    : {strat['budget']}")
    print(f"   KPIs      : {strat['kpi']}")
print('='*65)

## 7. Executive Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Customer Segmentation — Executive Summary', fontsize=18,
             fontweight='bold', y=0.98)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# ── Top KPI cards ─────────────────────────────────────────────────────────────
kpi_data = [
    ('Total Customers', f"{len(rfm):,}"),
    ('Total Revenue',   f"£{total_revenue:,.0f}"),
    ('Avg Order Value', f"£{rfm['Monetary'].mean():,.0f}"),
    ('Segments Found',  f"{N_CLUSTERS}"),
]

for i, (label, val) in enumerate(kpi_data):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor('#F1EFE8')
    ax.text(0.5, 0.65, val, ha='center', va='center',
            fontsize=22, fontweight='bold', transform=ax.transAxes, color='#2C2C2A')
    ax.text(0.5, 0.25, label, ha='center', va='center',
            fontsize=10, transform=ax.transAxes, color='#5F5E5A')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

# ── Revenue by segment (bar) ─────────────────────────────────────────────────
ax_rev = fig.add_subplot(gs[1, :2])
segs_ordered = [s for s in PERSONAS.keys() if s in seg_summary.index]
rev_vals   = [seg_summary.loc[s, 'Total_Revenue'] for s in segs_ordered]
rev_colors = [PERSONAS[s]['color'] for s in segs_ordered]
bars = ax_rev.bar(segs_ordered, rev_vals, color=rev_colors, edgecolor='white')
ax_rev.set_title('Total Revenue by Segment')
ax_rev.set_ylabel('Revenue (£)')
ax_rev.tick_params(axis='x', rotation=20)
ax_rev.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1e3:.0f}K'))
for bar, val in zip(bars, rev_vals):
    ax_rev.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f'£{val/1e3:.0f}K', ha='center', fontsize=9)

# ── Customer size (pie) ───────────────────────────────────────────────────────
ax_pie = fig.add_subplot(gs[1, 2:])
cust_vals = [seg_summary.loc[s, 'Customers'] for s in segs_ordered]
wedges, _, autotexts = ax_pie.pie(
    cust_vals, labels=segs_ordered, colors=rev_colors,
    autopct='%1.0f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5},
    pctdistance=0.75
)
for wt in autotexts:
    wt.set_fontsize(8)
ax_pie.set_title('Customer Distribution')

# ── RFM score distribution by segment ────────────────────────────────────────
ax_rfm = fig.add_subplot(gs[2, :])
for seg in segs_ordered:
    subset = rfm[rfm['Segment'] == seg]['RFM_Score']
    color  = PERSONAS[seg]['color']
    ax_rfm.hist(subset, bins=15, alpha=0.55, color=color,
                label=seg, edgecolor='white')

ax_rfm.set_title('RFM Score Distribution by Segment')
ax_rfm.set_xlabel('RFM Score (3=worst, 15=best)')
ax_rfm.set_ylabel('Customers')
ax_rfm.legend(fontsize=9, loc='upper left')

plt.savefig('../outputs/figures/executive_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved.')

## 8. Export Final Deliverable

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)

# ── Full customer table with segment ─────────────────────────────────────────
final_cols = [
    'Customer ID', 'Recency', 'Frequency', 'Monetary',
    'R_Score', 'F_Score', 'M_Score', 'RFM_Score',
    'Cluster', 'Segment', 'Segment_RuleBased'
]
final_df = rfm[final_cols].copy()
final_df.to_csv('../outputs/segments_final.csv', index=False)

# ── Segment strategy summary ─────────────────────────────────────────────────
strategy_rows = []
for seg in segs_ordered:
    if seg not in seg_summary.index or seg not in STRATEGY:
        continue
    row   = seg_summary.loc[seg]
    strat = STRATEGY[seg]
    strategy_rows.append({
        'Segment':          seg,
        'Customers':        int(row['Customers']),
        'Revenue_Share_%':  row['Revenue_Share_%'],
        'Avg_Recency_Days': row['Avg_Recency'],
        'Avg_Orders':       row['Avg_Frequency'],
        'Avg_Spend_GBP':    row['Avg_Monetary'],
        'Goal':             strat['goal'],
        'Channels':         ', '.join(strat['channels']),
        'Key_Message':      strat['message'],
        'Budget_Priority':  strat['budget'],
        'KPIs':             strat['kpi'],
    })

strategy_df = pd.DataFrame(strategy_rows)
strategy_df.to_csv('../outputs/segment_strategy.csv', index=False)

print('Exported:')
print(f'  outputs/segments_final.csv    ({len(final_df):,} customers)')
print(f'  outputs/segment_strategy.csv  ({len(strategy_df)} segments)')

In [ ]:
# Preview final output
print('\nFinal customer table (sample):')
print(final_df.head(10).to_string(index=False))

print('\nSegment strategy summary:')
print(strategy_df[['Segment','Customers','Revenue_Share_%','Goal','Budget_Priority']].to_string(index=False))

## ✅ Project Complete — Final Summary

### What we built

| Notebook | Output |
|----------|--------|
| 01 EDA | Clean transaction data + customer summary |
| 02 RFM | RFM scores (1–5) per customer + rule-based labels |
| 03 Clustering | K-Means clusters with optimal K via Elbow + Silhouette |
| 04 Insights | Named segments + personas + marketing strategy |

### Business value delivered

- **Identified** the exact customers generating 80% of revenue
- **Flagged** at-risk high-value customers for immediate action
- **Designed** 5 distinct marketing strategies — one per segment
- **Quantified** the revenue opportunity in each segment
- **Delivered** a clean CSV that plugs straight into any CRM or email tool

### Skills demonstrated
- Data cleaning & feature engineering with pandas
- RFM modeling (business + statistical)
- K-Means clustering with scikit-learn
- Model evaluation (Elbow, Silhouette)
- PCA dimensionality reduction
- Business storytelling through visualization
- Translating data findings into actionable marketing strategy